In [46]:
from google.colab import files

uploaded = files.upload()


Saving ted_talks_eda_ready.csv to ted_talks_eda_ready (1).csv


In [47]:
import pandas as pd
import numpy as np
import os

file_path = "/content/ted_talks_eda_ready.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        "Please upload ted_talks_eda_ready.csv first."
    )

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Number of rows: 4005
Number of columns: 30

Column names:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'combined_text', 'topic_list', 'topic_count', 'title_word_count', 'description_word_count', 'transcript_word_count', 'about_speakers_word_count', 'available_lang_list', 'available_lang_count', 'related_talks_list', 'related_talks_count']


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,topic_list,topic_count,title_word_count,description_word_count,transcript_word_count,about_speakers_word_count,available_lang_list,available_lang_count,related_talks_list,related_talks_count
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25 00:00:00+00:00,2006-06-27 00:00:00+00:00,TED2006,...,"['alternative energy', 'cars', 'climate change...",9,4,39,2153,24,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",43,"[""{243: 'New thinking on the climate crisis'"",...",6
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22 00:00:00+00:00,2006-06-27 00:00:00+00:00,TED2006,...,"['Africa', 'Asia', 'Google', 'demo', 'economic...",11,6,26,3174,35,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",49,"['{2056: ""Own your body\'s data""', ""2296: 'A v...",6
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24 00:00:00+00:00,2006-06-27 00:00:00+00:00,TED2006,...,"['computers', 'entertainment', 'interface desi...",9,2,31,3371,45,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",26,"[""{1725: '10 top time-saving tech tips'"", ""227...",6
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26 00:00:00+00:00,2006-06-27 00:00:00+00:00,TED2006,...,"['MacArthur grant', 'activism', 'business', 'c...",9,3,31,3071,32,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",35,"[""{1041: '3 stories of local eco-entrepreneurs...",6
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25 00:00:00+00:00,2006-06-27 00:00:00+00:00,TED2006,...,"['children', 'creativity', 'culture', 'dance',...",7,4,21,3170,32,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",62,"[""{865: 'Bring on the learning revolution!'"", ...",7


In [48]:
required_columns = [
    "title",
    "topics",
    "description",
    "transcript"
]

print("Text column verification:")

for col in required_columns:
    if col in df.columns:
        print(f"✓ {col}: available")
    else:
        print(f"✗ {col}: missing")

print("\nCombined text column available:",
      "combined_text" in df.columns)

Text column verification:
✓ title: available
✓ topics: available
✓ description: available
✓ transcript: available

Combined text column available: True


In [49]:

import pandas as pd
import numpy as np
import re

# Make sure the required text columns exist
text_columns = ["title", "topics", "description", "transcript"]

for col in text_columns:
    if col not in df.columns:
        df[col] = ""

# Replace missing values with empty strings
for col in text_columns:
    df[col] = df[col].fillna("").astype(str)

# Create combined text for recommendations
df["combined_text"] = (
    df["title"] + " " +
    df["topics"] + " " +
    df["description"] + " " +
    df["transcript"]
)

# Clean extra spaces and line breaks
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["combined_text"] = df["combined_text"].apply(clean_text)

# Check the results
print("Text preprocessing completed!")
print("Total TED Talks:", len(df))
print("Missing combined text:", (df["combined_text"].str.len() == 0).sum())

print("\nExample combined text:")
print(df.loc[0, "combined_text"][:500])


Text preprocessing completed!
Total TED Talks: 4005
Missing combined text: 0

Example combined text:
averting the climate crisis ['alternative energy', 'cars', 'climate change', 'culture', 'environment', 'global issues', 'science', 'sustainability', 'technology'] with the same humor and humanity he exuded in "an inconvenient truth," al gore spells out 15 ways that individuals can address climate change immediately, from buying a hybrid to inventing a new, hotter brand name for global warming. thank you so much, chris. and it's truly a great honor to have the opportunity to come to this stage tw


In [50]:

from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)

# Convert the combined text into a sparse numerical matrix
tfidf_matrix = tfidf.fit_transform(df["combined_text"])

# Display results
print("TF-IDF vectorization completed successfully!")
print("Number of TED Talks:", tfidf_matrix.shape[0])
print("Number of text features:", tfidf_matrix.shape[1])
print("Matrix type:", type(tfidf_matrix))
print("Vocabulary size:", len(tfidf.vocabulary_))

# Display a few vocabulary terms
print("\nSample vocabulary terms:")
print(list(tfidf.vocabulary_.keys())[:20])


TF-IDF vectorization completed successfully!
Number of TED Talks: 4005
Number of text features: 50000
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Vocabulary size: 50000

Sample vocabulary terms:
['climate', 'crisis', 'alternative', 'energy', 'cars', 'change', 'culture', 'environment', 'global', 'issues', 'science', 'sustainability', 'technology', 'humor', 'humanity', 'inconvenient', 'truth', 'al', 'gore', 'spells']


In [51]:

import joblib
import os

# Create a folder for project artifacts
os.makedirs("/content/ted_recommender_artifacts", exist_ok=True)

# Save the fitted TF-IDF vectorizer
joblib.dump(
    tfidf,
    "/content/ted_recommender_artifacts/ted_tfidf_vectorizer.joblib"
)

# Save the TF-IDF sparse matrix
joblib.dump(
    tfidf_matrix,
    "/content/ted_recommender_artifacts/ted_tfidf_matrix.joblib"
)

# Save the dataset in the same row order as the matrix
df.to_csv(
    "/content/ted_recommender_artifacts/ted_model_dataset.csv",
    index=False
)

print("All artifacts saved successfully!")
print("\nSaved files:")

for file in os.listdir("/content/ted_recommender_artifacts"):
    path = os.path.join("/content/ted_recommender_artifacts", file)
    print(f"{file} — {os.path.getsize(path) / (1024 * 1024):.2f} MB")


All artifacts saved successfully!

Saved files:
ted_model_dataset.csv — 86.21 MB
ted_tfidf_matrix.joblib — 17.15 MB
ted_tfidf_vectorizer.joblib — 1.72 MB


In [52]:

from sklearn.metrics.pairwise import cosine_similarity

def recommend_talks(title, top_n=5):
    # Find an exact title match, ignoring uppercase/lowercase
    matches = df[
        df["title"].astype(str).str.strip().str.casefold()
        == title.strip().casefold()
    ]

    if matches.empty:
        print("Title not found in the dataset.")
        print("\nPossible matching titles:")

        suggestions = df[
            df["title"].astype(str).str.contains(
                title, case=False, na=False, regex=False
            )
        ]["title"].drop_duplicates().head(10)

        if suggestions.empty:
            print("No similar titles found. Try another search.")
        else:
            for suggestion in suggestions:
                print("-", suggestion)

        return None

    # Use the first matching talk
    talk_index = matches.index[0]

    # Compare this talk with every talk in the dataset
    similarities = cosine_similarity(
        tfidf_matrix[talk_index],
        tfidf_matrix
    ).flatten()

    # Sort by similarity, highest first
    ranked_indices = similarities.argsort()[::-1]

    # Exclude the selected talk itself
    ranked_indices = [
        idx for idx in ranked_indices
        if idx != talk_index
    ]

    # Select the top recommendations
    recommended_indices = ranked_indices[:top_n]

    # Prepare recommendation results
    result_columns = [
        col for col in [
            "title", "speaker_1", "topics", "description", "url"
        ]
        if col in df.columns
    ]

    recommendations = df.iloc[recommended_indices][
        result_columns
    ].copy()

    recommendations["similarity_score"] = [
        round(float(similarities[idx]), 4)
        for idx in recommended_indices
    ]

    return recommendations.reset_index(drop=True)


print("Recommendation engine created successfully!")


Recommendation engine created successfully!


In [53]:

# Display a few titles available in the dataset
print("Sample TED Talk titles:")
print(df["title"].head(10).to_string(index=False))

# Test with the first talk in your dataset
sample_title = df.iloc[0]["title"]

print("\nSelected TED Talk:")
print(sample_title)

print("\nRecommended TED Talks:")
recommendations = recommend_talks(sample_title, top_n=5)

if recommendations is not None:
    display(recommendations)


Sample TED Talk titles:
                       Averting the climate crisis
                   The best stats you've ever seen
                                  Simplicity sells
                               Greening the ghetto
                       Do schools kill creativity?
            Behind the design of Seattle's library
                                 Letting go of God
Let's teach religion -- all religion -- in schools
                                 A life of purpose
                     My wish: A global day of film

Selected TED Talk:
Averting the climate crisis

Recommended TED Talks:


,title,speaker_1,topics,description,url,similarity_score
0,New thinking on the climate crisis,Al Gore,"['activism', 'climate change', 'global issues'...",In this brand-new slideshow (premiering on TED...,https://www.ted.com/talks/al_gore_new_thinking...,0.1609
1,How we can turn the tide on climate,Chris Anderson,"['climate change', 'environment', 'activism', ...","Witness the unveiling of Countdown, a major gl...",https://www.ted.com/talks/christiana_figueres_...,0.1525
2,The case for optimism on climate change,Al Gore,"['climate change', 'weather', 'science', 'alte...",Why is Al Gore optimistic about climate change...,https://www.ted.com/talks/al_gore_the_case_for...,0.1490
3,What comes after An Inconvenient Truth?,Al Gore,"['alternative energy', 'climate change', 'ener...","At TED2009, Al Gore presents updated slides fr...",https://www.ted.com/talks/al_gore_what_comes_a...,0.1489
4,Salvation (and profit) in greentech,John Doerr,"['climate change', 'environment', 'global issu...","""I don't think we're going to make it,"" John D...",https://www.ted.com/talks/john_doerr_salvation...,0.1353


In [54]:

def recommend_by_interest(query, top_n=5):
    # Validate the user's input
    if not isinstance(query, str) or not query.strip():
        print("Please enter a topic or interest.")
        return None

    # Convert the user's query using the existing TF-IDF vocabulary
    query_vector = tfidf.transform([query.strip().lower()])

    # Check whether the query contains known vocabulary
    if query_vector.nnz == 0:
        print("No recognizable terms found in the TF-IDF vocabulary.")
        print("Try a topic such as climate change, technology, or education.")
        return None

    # Compare the query with all TED Talks
    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Rank talks by similarity
    ranked_indices = similarities.argsort()[::-1][:top_n]

    # Prepare the results
    result_columns = [
        col for col in [
            "title", "speaker_1", "topics", "description", "url"
        ]
        if col in df.columns
    ]

    recommendations = df.iloc[ranked_indices][result_columns].copy()

    recommendations["similarity_score"] = [
        round(float(similarities[idx]), 4)
        for idx in ranked_indices
    ]

    return recommendations.reset_index(drop=True)


print("Interest-based recommendation function created!")


Interest-based recommendation function created!


In [55]:

user_interest = "artificial intelligence and machine learning"

print("User interest:", user_interest)
print("\nRecommended TED Talks:")

interest_recommendations = recommend_by_interest(
    user_interest,
    top_n=5
)

if interest_recommendations is not None:
    display(interest_recommendations)


User interest: artificial intelligence and machine learning

Recommended TED Talks:


,title,speaker_1,topics,description,url,similarity_score
0,"How AI can enhance our memory, work and social...",Tom Gruber,"['AI', 'collaboration', 'computers', 'intellig...",How smart can our machines make us? Tom Gruber...,https://www.ted.com/talks/tom_gruber_how_ai_ca...,0.1604
1,4 ways to build a human company in the age of ...,Tim Leberecht,"['business', 'beauty', 'collaboration', 'goal-...",In the face of artificial intelligence and mac...,https://www.ted.com/talks/tim_leberecht_4_ways...,0.1598
2,The jobs we'll lose to machines -- and the one...,Anthony Goldbloom,"['AI', 'collaboration', 'intelligence', 'data'...",Machine learning isn't just for simple tasks l...,https://www.ted.com/talks/anthony_goldbloom_th...,0.1576
3,How AI is making it easier to diagnose disease,Pratik Shah,"['medicine', 'TED Fellows', 'AI', 'technology'...",Today's AI algorithms require tens of thousand...,https://www.ted.com/talks/pratik_shah_how_ai_i...,0.1533
4,"How we can build AI to help humans, not hurt us",Margaret Mitchell,"['algorithm', 'AI', 'communication', 'collabor...","As a research scientist at Google, Margaret Mi...",https://www.ted.com/talks/margaret_mitchell_ho...,0.1360


In [56]:

test_interests = [
    "climate change and environment",
    "artificial intelligence and machine learning",
    "education and creativity",
    "healthcare and medical technology",
    "leadership and personal growth"
]

for interest in test_interests:
    print("\n" + "=" * 70)
    print("USER INTEREST:", interest)
    print("=" * 70)

    results = recommend_by_interest(interest, top_n=5)

    if results is not None:
        display(
            results[["title", "speaker_1", "similarity_score"]]
        )



USER INTEREST: climate change and environment


,title,speaker_1,similarity_score
0,"Global warming's theme song, ""Manhattan in Jan...",Jill Sobule,0.1733
1,The science behind a climate headline,Rachel Pike,0.1406
2,The disarming case to act right now on climate...,Greta Thunberg,0.1185
3,The emergent patterns of climate change,Gavin Schmidt,0.1121
4,3 psychological tricks to help you save money,Wendy De La Rosa,0.1055



USER INTEREST: artificial intelligence and machine learning


,title,speaker_1,similarity_score
0,"How AI can enhance our memory, work and social...",Tom Gruber,0.1604
1,4 ways to build a human company in the age of ...,Tim Leberecht,0.1598
2,The jobs we'll lose to machines -- and the one...,Anthony Goldbloom,0.1576
3,How AI is making it easier to diagnose disease,Pratik Shah,0.1533
4,"How we can build AI to help humans, not hurt us",Margaret Mitchell,0.1360



USER INTEREST: education and creativity


,title,speaker_1,similarity_score
0,Do the green thing,Andy Hobsbawm,0.1392
1,The left brain vs. right brain myth,Elizabeth Waters,0.1041
2,Can robots be creative?,Gil Weinberg,0.0859
3,Do schools kill creativity?,Sir Ken Robinson,0.0830
4,"How to be ""Team Human"" in the digital future",Douglas Rushkoff,0.0794



USER INTEREST: healthcare and medical technology


,title,speaker_1,similarity_score
0,A second opinion on developmental disorders,Aditi Shankardass,0.1183
1,Are we over-medicalized?,Ivan Oransky,0.0998
2,How sharks could inspire a new generation of m...,Ethan Mann,0.0770
3,What doctors can learn from each other,Stefan Larsson,0.0697
4,A doctor's case for medical marijuana,David Casarett,0.0691



USER INTEREST: leadership and personal growth


,title,speaker_1,similarity_score
0,What it takes to be a great leader,Roselinde Torres,0.0891
1,What Bruce Lee can teach us about living fully,Shannon Lee,0.0880
2,The best gift I ever survived,Stacey Kramer,0.0837
3,How to break bad management habits before they...,Elizabeth Lyle,0.0799
4,How diversity makes teams more innovative,Rocío Lorenzo,0.0793


In [57]:

# Inspect the raw related_talks values
print("Related talks column exists:", "related_talks" in df.columns)

if "related_talks" in df.columns:
    print("\nData type:")
    print(df["related_talks"].dtype)

    print("\nFirst 10 raw values:")
    for i, value in enumerate(df["related_talks"].head(10)):
        print(f"\nRow {i}:")
        print(repr(value)[:1000])

    print("\nMissing related_talks values:",
          df["related_talks"].isna().sum())
else:
    print("The related_talks column is unavailable.")


Related talks column exists: True

Data type:
object

First 10 raw values:

Row 0:
'{243: \'New thinking on the climate crisis\', 547: \'The business logic of sustainability\', 2093: \'The state of the climate — and what we might do about it\', 54715: \'How we can turn the tide on climate\', 29968: \'The most important thing you can do to fight climate change: talk about it\', 2339: "Climate change is happening. Here\'s how we adapt"}'

Row 1:
'{2056: "Own your body\'s data", 2296: \'A visual history of human knowledge\', 620: \'Let my dataset change your mindset\', 2806: "Doesn\'t everyone deserve a chance at a good life?", 2560: \'How Africa can keep rising\', 1418: "Let\'s put birth control back on the agenda"}'

Row 2:
"{1725: '10 top time-saving tech tips', 2274: 'The first secret of design is ... noticing', 172: 'Designing for simplicity', 2664: 'Meet the inventor of the electronic spreadsheet', 2464: 'The mind behind Linux', 1347: 'The secret structure of great talks'}"

Row 3:


In [58]:

import ast

# Parse related_talks strings into dictionaries
def parse_related_talks(value):
    if isinstance(value, dict):
        return value

    if not isinstance(value, str) or not value.strip():
        return {}

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, dict) else {}
    except (ValueError, SyntaxError):
        return {}

df["related_talks_parsed"] = df["related_talks"].apply(
    parse_related_talks
)

# Convert talk IDs into a consistent type
def normalize_related_ids(related_dict):
    normalized = set()

    for talk_id in related_dict.keys():
        try:
            normalized.add(int(talk_id))
        except (ValueError, TypeError):
            continue

    return normalized

df["related_talk_ids"] = df["related_talks_parsed"].apply(
    normalize_related_ids
)

# Compare related IDs against the dataset's talk_id values
dataset_ids = set(
    pd.to_numeric(df["talk_id"], errors="coerce")
      .dropna()
      .astype(int)
)

all_related_ids = set().union(*df["related_talk_ids"])

matched_ids = all_related_ids.intersection(dataset_ids)
unmatched_ids = all_related_ids - dataset_ids

print("Related-talk parsing completed!")
print("Talks in dataset:", len(df))
print("Unique related IDs:", len(all_related_ids))
print("Related IDs found in dataset:", len(matched_ids))
print("Related IDs not found in dataset:", len(unmatched_ids))

if all_related_ids:
    print(
        "Related-ID match rate:",
        f"{len(matched_ids) / len(all_related_ids) * 100:.2f}%"
    )

print("\nExample parsed dictionary:")
print(df.loc[0, "related_talks_parsed"])


Related-talk parsing completed!
Talks in dataset: 4005
Unique related IDs: 3822
Related IDs found in dataset: 3766
Related IDs not found in dataset: 56
Related-ID match rate: 98.53%

Example parsed dictionary:
{243: 'New thinking on the climate crisis', 547: 'The business logic of sustainability', 2093: 'The state of the climate — and what we might do about it', 54715: 'How we can turn the tide on climate', 29968: 'The most important thing you can do to fight climate change: talk about it', 2339: "Climate change is happening. Here's how we adapt"}


In [59]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Map each talk ID to its dataframe row
talk_id_to_index = {
    int(talk_id): idx
    for idx, talk_id in enumerate(
        pd.to_numeric(df["talk_id"], errors="coerce")
    )
    if pd.notna(talk_id)
}

def evaluate_at_k(k=5, max_talks=500):
    precisions = []
    recalls = []
    evaluated = 0

    # Evaluate talks that have at least one known related talk
    eligible_indices = [
        idx for idx, related_ids in enumerate(df["related_talk_ids"])
        if related_ids
    ]

    # Use a fixed sample for reproducible results
    rng = np.random.default_rng(42)
    sample_indices = rng.choice(
        eligible_indices,
        size=min(max_talks, len(eligible_indices)),
        replace=False
    )

    for idx in sample_indices:
        source_id = int(df.iloc[idx]["talk_id"])

        # Keep only known related talks that exist in the dataset
        relevant_ids = {
            rid for rid in df.iloc[idx]["related_talk_ids"]
            if rid in talk_id_to_index and rid != source_id
        }

        if not relevant_ids:
            continue

        # Generate top-k recommendations from this talk
        similarities = cosine_similarity(
            tfidf_matrix[idx], tfidf_matrix
        ).ravel()

        similarities[idx] = -1
        top_indices = np.argsort(similarities)[::-1][:k]

        recommended_ids = {
            int(df.iloc[j]["talk_id"])
            for j in top_indices
        }

        hits = len(recommended_ids.intersection(relevant_ids))

        precisions.append(hits / k)
        recalls.append(hits / len(relevant_ids))
        evaluated += 1

    if evaluated == 0:
        print("No eligible talks could be evaluated.")
        return None

    results = {
        f"Precision@{k}": float(np.mean(precisions)),
        f"Recall@{k}": float(np.mean(recalls)),
        "Talks evaluated": evaluated
    }

    return results


evaluation_results = evaluate_at_k(k=5, max_talks=500)

print("Evaluation results:")
print(evaluation_results)


Evaluation results:
{'Precision@5': 0.312, 'Recall@5': 0.2621666666666666, 'Talks evaluated': 500}


In [60]:

import pandas as pd
import os

# Convert evaluation results into a table
evaluation_df = pd.DataFrame([
    {
        "Metric": "Precision@5",
        "Score": evaluation_results["Precision@5"]
    },
    {
        "Metric": "Recall@5",
        "Score": evaluation_results["Recall@5"]
    },
    {
        "Metric": "Talks Evaluated",
        "Score": evaluation_results["Talks evaluated"]
    }
])

# Create the artifact folder if needed
artifact_dir = "/content/ted_recommender_artifacts"
os.makedirs(artifact_dir, exist_ok=True)

# Save metrics as CSV
evaluation_path = os.path.join(
    artifact_dir, "ted_recommendation_evaluation.csv"
)
evaluation_df.to_csv(evaluation_path, index=False)

display(evaluation_df)
print("\nEvaluation results saved to:", evaluation_path)


,Metric,Score
0,Precision@5,0.312000
1,Recall@5,0.262167
2,Talks Evaluated,500.000000



Evaluation results saved to: /content/ted_recommender_artifacts/ted_recommendation_evaluation.csv


In [61]:

import ast
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Convert topic values into lists safely
def parse_topics(value):
    if isinstance(value, list):
        return [str(topic).strip().lower() for topic in value]

    if not isinstance(value, str) or not value.strip():
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(topic).strip().lower() for topic in parsed]
    except (ValueError, SyntaxError):
        pass

    # Fallback for comma-separated topic strings
    return [
        topic.strip().lower()
        for topic in value.split(",")
        if topic.strip()
    ]

df["topics_list"] = df["topics"].apply(parse_topics)

def recommend_with_topic(query, topic, top_n=5):
    if not isinstance(query, str) or not query.strip():
        print("Please enter a search query.")
        return None

    if not isinstance(topic, str) or not topic.strip():
        print("Please enter a topic.")
        return None

    query_vector = tfidf.transform([query.strip().lower()])

    if query_vector.nnz == 0:
        print("No recognizable query terms found.")
        return None

    similarities = cosine_similarity(
        query_vector, tfidf_matrix
    ).ravel()

    # Normalize the selected topic
    selected_topic = topic.strip().lower()

    # Match talks whose parsed topic contains the selected topic
    topic_match = df["topics_list"].apply(
        lambda topics: any(
            selected_topic in item for item in topics
        )
    ).to_numpy()

    # Keep only talks matching the selected topic
    candidate_indices = np.where(topic_match)[0]

    if len(candidate_indices) == 0:
        print("No talks found for this topic.")
        return None

    # Rank topic-matched talks by their text similarity
    ranked = candidate_indices[
        np.argsort(similarities[candidate_indices])[::-1]
    ][:top_n]

    result_columns = [
        col for col in [
            "title", "speaker_1", "topics", "description", "url"
        ]
        if col in df.columns
    ]

    results = df.iloc[ranked][result_columns].copy()
    results["similarity_score"] = [
        round(float(similarities[idx]), 4)
        for idx in ranked
    ]

    return results.reset_index(drop=True)

print("Topic-filtered recommendation function created!")


Topic-filtered recommendation function created!


In [62]:

display(
    recommend_with_topic(
        query="climate change and environmental protection",
        topic="climate change",
        top_n=5
    )
)


,title,speaker_1,topics,description,url,similarity_score
0,A small country with big ideas to get rid of f...,Monica Araya,"['alternative energy', 'activism', 'climate ch...",How do we build a society without fossil fuels...,https://www.ted.com/talks/monica_araya_a_small...,0.1603
1,A personal air-quality tracker that lets you k...,Romain Lacombe,"['TED Fellows', 'technology', 'pollution', 'en...",How often do you think about the air you're br...,https://www.ted.com/talks/romain_lacombe_a_per...,0.1330
2,How one scientist took on the chemical industry,Mark Lytle,"['TED-Ed', 'agriculture', 'education', 'animat...","In 1958, after receiving a letter describing t...",https://www.ted.com/talks/mark_lytle_how_one_s...,0.1120
3,How China is (and isn't) fighting pollution an...,Angel Hsu,"['climate change', 'sustainability', 'green', ...",China is the world's biggest polluter -- and n...,https://www.ted.com/talks/angel_hsu_how_china_...,0.1113
4,Why I must speak out about climate change,James Hansen,"['big problems', 'climate change', 'environmen...",Top climate scientist James Hansen tells the s...,https://www.ted.com/talks/james_hansen_why_i_m...,0.0964


In [63]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Prepare shorter, more focused text
df["title_topics_text"] = (
    df["title"].fillna("").astype(str) + " " +
    df["topics"].fillna("").astype(str) + " " +
    df["topics"].fillna("").astype(str)
).str.lower().str.strip()

# Create a second TF-IDF model for comparison
tfidf_focused = TfidfVectorizer(
    stop_words="english",
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)

focused_matrix = tfidf_focused.fit_transform(
    df["title_topics_text"]
)

print("Focused TF-IDF model created!")
print("Talks:", focused_matrix.shape[0])
print("Features:", focused_matrix.shape[1])


Focused TF-IDF model created!
Talks: 4005
Features: 8171


In [64]:

def evaluate_matrix_at_k(matrix, k=5, max_talks=500, seed=42):
    eligible = [
        idx for idx, related_ids in enumerate(df["related_talk_ids"])
        if any(
            rid in talk_id_to_index and rid != int(df.iloc[idx]["talk_id"])
            for rid in related_ids
        )
    ]

    rng = np.random.default_rng(seed)
    sample = rng.choice(
        eligible,
        size=min(max_talks, len(eligible)),
        replace=False
    )

    precisions = []
    recalls = []

    for idx in sample:
        source_id = int(df.iloc[idx]["talk_id"])

        relevant = {
            rid for rid in df.iloc[idx]["related_talk_ids"]
            if rid in talk_id_to_index and rid != source_id
        }

        similarities = cosine_similarity(
            matrix[idx], matrix
        ).ravel()

        similarities[idx] = -1
        top_indices = np.argsort(similarities)[::-1][:k]

        recommended_ids = {
            int(df.iloc[j]["talk_id"]) for j in top_indices
        }

        hits = len(recommended_ids.intersection(relevant))
        precisions.append(hits / k)
        recalls.append(hits / len(relevant))

    return {
        f"Precision@{k}": np.mean(precisions),
        f"Recall@{k}": np.mean(recalls),
        "Talks evaluated": len(precisions)
    }


baseline_results = evaluate_matrix_at_k(
    tfidf_matrix, k=5, max_talks=500, seed=42
)

focused_results = evaluate_matrix_at_k(
    focused_matrix, k=5, max_talks=500, seed=42
)

comparison = pd.DataFrame([
    {"Model": "Original: full text", **baseline_results},
    {"Model": "Focused: title + topics", **focused_results}
])

display(comparison)


,Model,Precision@5,Recall@5,Talks evaluated
0,Original: full text,0.3108,0.261267,500
1,Focused: title + topics,0.1408,0.119333,500


In [67]:
def validate_recommendation_output():
    print("WEEK 4 FINAL VALIDATION")
    print("-" * 40)

    # Test a valid interest query
    results = recommend_by_interest(
        "artificial intelligence and machine learning",
        top_n=5
    )

    assert isinstance(results, pd.DataFrame), \
        "Output should be a DataFrame"

    assert len(results) <= 5, \
        "Returned more recommendations than requested"

    if not results.empty:
        assert results["title"].notna().all(), \
            "A recommendation has a missing title"

        assert not results["title"].duplicated().any(), \
            "Duplicate recommendation titles found"

        assert results["similarity_score"].between(0, 1).all(), \
            "Similarity scores are outside the expected range"

    print("Valid query test: PASSED")
    print("Recommendations returned:", len(results))

    # Test invalid recommendation count
    for n in [0, -1]:
        try:
            recommend_by_interest("education", top_n=n)
            print(f"top_n={n}: handled without an exception")
        except (ValueError, TypeError) as e:
            print(f"top_n={n}: rejected safely ({e})")

    print("\nValidation completed.")

validate_recommendation_output()

WEEK 4 FINAL VALIDATION
----------------------------------------
Valid query test: PASSED
Recommendations returned: 5
top_n=0: handled without an exception
top_n=-1: handled without an exception

Validation completed.
